In [23]:
from pathlib import Path
import os, re, json, time, hashlib, platform
from datetime import datetime
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from IPython.display import display

PROJECT = Path('/content/UTS-DAM-Pariwisata-NTB')
RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
META = PROJECT / 'metadata'
for folder in [RAW, PROCESSED, META]:
    folder.mkdir(parents=True, exist_ok=True)

# Memperbarui email kampus sesuai dengan yang Anda gunakan di sel validasi
CAMPUS_EMAIL = 'mikiantul@gmail.com'
USER_AGENT = f'UTS-DAM-Student-Project ({CAMPUS_EMAIL})'
print('Folder proyek:', PROJECT)
print('Email User-Agent sudah diisi:', CAMPUS_EMAIL != 'GANTI_DENGAN_EMAIL_KAMPUS_PRIBADI')

Folder proyek: /content/UTS-DAM-Pariwisata-NTB
Email User-Agent sudah diisi: True


In [6]:
from google.colab import files
import shutil

dataset_path = RAW / 'dataset_kunjungan_wisatawan_ntb_asli.xlsx'
if not dataset_path.exists():
    print('Upload file XLSX dataset dari Satu Data NTB.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('File dataset belum diunggah.')
    uploaded_name = next(iter(uploaded))
    dataset_path.write_bytes(uploaded[uploaded_name])  # simpan byte asli tanpa edit
else:
    print('Dataset ditemukan:', dataset_path)

print('Ukuran file (byte):', dataset_path.stat().st_size)


Upload file XLSX dataset dari Satu Data NTB.


Saving data_20261125011910-jumlah-kunjungan-wisatawan-berdasarkan-jenis-wisatawan (1).xlsx to data_20261125011910-jumlah-kunjungan-wisatawan-berdasarkan-jenis-wisatawan (1).xlsx
Ukuran file (byte): 16796


In [7]:
# Baca Excel dengan sheet_name yang ditentukan dan tipe awal yang aman.
df_raw = pd.read_excel(
    dataset_path,
    sheet_name='Worksheet',
    dtype={'Kode provinsi': 'string', 'Jumlah (Orang)': 'string'}
)
print('Ukuran dataset:', df_raw.shape)
print('Kolom:', df_raw.columns.tolist())
display(df_raw.head())
display(df_raw.dtypes.to_frame('tipe_data_awal'))


Ukuran dataset: (300, 7)
Kolom: ['#', 'Kode provinsi', 'Nama Provinsi', 'Kategori Wisatawan', 'Jumlah (Orang)', 'Tahun', 'Bulan']


,#,Kode provinsi,Nama Provinsi,Kategori Wisatawan,Jumlah (Orang),Tahun,Bulan
0,1,52,Nusa Tenggara Barat,Wisatawan Nusantara,"132,472",2026,Juni
1,2,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"88,315",2026,Juni
2,3,52,Nusa Tenggara Barat,Wisatawan Nusantara,"122,674",2026,Mei
3,4,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"81,783",2026,Mei
4,5,52,Nusa Tenggara Barat,Wisatawan Nusantara,"113,304",2026,April


,tipe_data_awal
#,int64
Kode provinsi,string[python]
Nama Provinsi,object
Kategori Wisatawan,object
Jumlah (Orang),string[python]
Tahun,int64
Bulan,object


In [8]:
# Salinan analisis; file asli di data/raw tidak disentuh.
df = df_raw.copy()
df.columns = ['nomor','kode_provinsi','nama_provinsi','kategori_wisatawan',
              'jumlah_orang','tahun','bulan']

df['jumlah'] = pd.to_numeric(
    df['jumlah_orang'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)
df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').astype('Int64')
df['kode_provinsi'] = pd.to_numeric(df['kode_provinsi'], errors='coerce').astype('Int64')
df['bulan'] = df['bulan'].astype(str).str.strip()

# Periksa ejaan kategori. Perbaikan hanya pada salinan analisis.
print('Kategori sebelum standardisasi:')
display(df['kategori_wisatawan'].value_counts(dropna=False))
df['kategori_wisatawan'] = df['kategori_wisatawan'].replace({
    'Wisatawan Mancaegara': 'Wisatawan Mancanegara'
})
print('Kategori setelah standardisasi:')
display(df['kategori_wisatawan'].value_counts(dropna=False))
display(df.head())

Kategori sebelum standardisasi:


,count
kategori_wisatawan,
Wisatawan Nusantara,150
Wisatawan Mancanegara,149
Wisatawan Mancaegara,1


Kategori setelah standardisasi:


,count
kategori_wisatawan,
Wisatawan Nusantara,150
Wisatawan Mancanegara,150


,nomor,kode_provinsi,nama_provinsi,kategori_wisatawan,jumlah_orang,tahun,bulan,jumlah
0,1,52,Nusa Tenggara Barat,Wisatawan Nusantara,"132,472",2026,Juni,132472
1,2,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"88,315",2026,Juni,88315
2,3,52,Nusa Tenggara Barat,Wisatawan Nusantara,"122,674",2026,Mei,122674
3,4,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"81,783",2026,Mei,81783
4,5,52,Nusa Tenggara Barat,Wisatawan Nusantara,"113,304",2026,April,113304


In [9]:
file_bytes = dataset_path.stat().st_size
file_mb = file_bytes / (1024**2)
annual_storage_bytes = file_bytes * 12 * 1.30

print('Jenis data: Terstruktur (tabular)')
print('Jumlah baris:', len(df))
print('Jumlah kolom:', len(df.columns))
print('Kolom analisis:', df.columns.tolist())
print('Tipe data analisis:')
display(df.dtypes.to_frame('dtype'))
print('Wilayah:', ', '.join(df['nama_provinsi'].dropna().astype(str).unique()))
print('Tahun:', int(df['tahun'].min()), 's.d.', int(df['tahun'].max()))
print('Bulan unik:', sorted(df['bulan'].dropna().unique().tolist()))
print('Satuan jumlah: orang')
print(f'Ukuran file: {file_bytes:,} byte = {file_mb:.4f} MB')
print(f'Estimasi storage 1 tahun: {annual_storage_bytes:,.0f} byte = {annual_storage_bytes/(1024**2):.4f} MB')
print('Asumsi: satu file dengan ukuran sama dikumpulkan setiap bulan (12 bulan), lalu ditambah cadangan 30%.')
print('Nilai kosong per kolom:')
display(df.isna().sum().to_frame('jumlah_kosong'))
print('Duplikasi baris penuh:', int(df.duplicated().sum()))

Jenis data: Terstruktur (tabular)
Jumlah baris: 300
Jumlah kolom: 8
Kolom analisis: ['nomor', 'kode_provinsi', 'nama_provinsi', 'kategori_wisatawan', 'jumlah_orang', 'tahun', 'bulan', 'jumlah']
Tipe data analisis:


,dtype
nomor,int64
kode_provinsi,Int64
nama_provinsi,object
kategori_wisatawan,object
jumlah_orang,string[python]
tahun,Int64
bulan,object
jumlah,int64


Wilayah: Nusa Tenggara Barat
Tahun: 2014 s.d. 2026
Bulan unik: ['Agustus', 'April', 'Desember', 'Februari', 'Januari', 'Juli', 'Juni', 'Maret', 'Mei', 'November', 'Oktober', 'September']
Satuan jumlah: orang
Ukuran file: 16,796 byte = 0.0160 MB
Estimasi storage 1 tahun: 262,018 byte = 0.2499 MB
Asumsi: satu file dengan ukuran sama dikumpulkan setiap bulan (12 bulan), lalu ditambah cadangan 30%.
Nilai kosong per kolom:


,jumlah_kosong
nomor,0
kode_provinsi,0
nama_provinsi,0
kategori_wisatawan,0
jumlah_orang,0
tahun,0
bulan,0
jumlah,0


Duplikasi baris penuh: 0


In [10]:
ethics = pd.DataFrame([
 ['1. Sumber dan tujuan','Satu Data NTB; digunakan untuk analisis akademik UTS','Sesuai','URL sumber utama dicatat pada provenance'],
 ['2. Lisensi dan ketentuan','Metadata sumber menampilkan License Not Specified','Perlu klarifikasi','Tidak mengklaim lisensi terbuka; atribusikan portal'],
 ['3. PII langsung','Tidak ditemukan nama/alamat/telepon individu pada struktur kolom yang diperiksa','Tidak teridentifikasi','Kolom: kode/nama provinsi, kategori, jumlah, tahun, bulan'],
 ['4. Quasi-identifier','Wilayah, tahun, bulan, kategori adalah atribut agregat','Risiko rendah','Tidak terdapat identitas individu dalam dataset agregat'],
 ['5. Minimisasi data','Analisis menggunakan variabel relevan: kategori, jumlah, tahun, bulan','Sesuai','File mentah dipisahkan dari salinan analisis'],
 ['6. Keamanan','Tidak menyimpan API key/token dalam notebook','Sesuai','API Open-Meteo yang digunakan tidak membutuhkan token'],
 ['7. Transparansi/provenance','URL, waktu akses, lisensi, nama file dan MD5 dicatat','Sesuai setelah sel provenance dijalankan','metadata/provenance.csv'],
 ['8. Kepatuhan API/scraping','Timeout, User-Agent email kampus, jeda, status code dan robots.txt diperiksa','Bersyarat','Bukti disimpan pada metadata dan output notebook'],
 ['9. Penyimpanan/publikasi','Raw tidak diubah; data turunan diberi sumber; tidak mengunggah rahasia','Sesuai','data/raw dan data/processed']
], columns=['butir','pemeriksaan','status','bukti_catatan'])
display(ethics)
ethics.to_csv(META/'data_ethics_checklist.csv', index=False)


,butir,pemeriksaan,status,bukti_catatan
0,1. Sumber dan tujuan,Satu Data NTB; digunakan untuk analisis akadem...,Sesuai,URL sumber utama dicatat pada provenance
1,2. Lisensi dan ketentuan,Metadata sumber menampilkan License Not Specified,Perlu klarifikasi,Tidak mengklaim lisensi terbuka; atribusikan p...
2,3. PII langsung,Tidak ditemukan nama/alamat/telepon individu p...,Tidak teridentifikasi,"Kolom: kode/nama provinsi, kategori, jumlah, t..."
3,4. Quasi-identifier,"Wilayah, tahun, bulan, kategori adalah atribut...",Risiko rendah,Tidak terdapat identitas individu dalam datase...
4,5. Minimisasi data,Analisis menggunakan variabel relevan: kategor...,Sesuai,File mentah dipisahkan dari salinan analisis
5,6. Keamanan,Tidak menyimpan API key/token dalam notebook,Sesuai,API Open-Meteo yang digunakan tidak membutuhka...
6,7. Transparansi/provenance,"URL, waktu akses, lisensi, nama file dan MD5 d...",Sesuai setelah sel provenance dijalankan,metadata/provenance.csv
7,8. Kepatuhan API/scraping,"Timeout, User-Agent email kampus, jeda, status...",Bersyarat,Bukti disimpan pada metadata dan output notebook
8,9. Penyimpanan/publikasi,Raw tidak diubah; data turunan diberi sumber; ...,Sesuai,data/raw dan data/processed


In [24]:
if CAMPUS_EMAIL == 'GANTI_DENGAN_EMAIL_KAMPUS_PRIBADI' or '@' not in CAMPUS_EMAIL:
    raise ValueError('Ganti CAMPUS_EMAIL dengan email kampus pribadi sebelum request.')

API_URL = 'https://archive-api.open-meteo.com/v1/archive'
api_params = {
    'latitude': -8.5833,
    'longitude': 116.1167,
    'start_date': '2024-01-01',
    'end_date': '2024-12-31',
    'daily': 'temperature_2m_mean,precipitation_sum,wind_speed_10m_max',
    'timezone': 'Asia/Makassar'
}
headers = {'User-Agent': USER_AGENT}

def request_with_retry(url, params=None, headers=None, timeout=30, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
            if response.status_code == 200:
                return response
            if response.status_code in (429, 500, 502, 503, 504):
                time.sleep((attempt + 1) * 2)
                continue
            response.raise_for_status()
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep((attempt + 1) * 2)
    raise RuntimeError('Request gagal setelah beberapa percobaan.')

time.sleep(1)
api_response = request_with_retry(API_URL, params=api_params, headers=headers)
api_json = api_response.json()
api_raw_path = RAW/'open_meteo_mataram_2024_raw.json'
api_raw_path.write_text(json.dumps(api_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('Status HTTP:', api_response.status_code)
print('URL request:', api_response.url)
print('File JSON mentah:', api_raw_path)

Status HTTP: 200
URL request: https://archive-api.open-meteo.com/v1/archive?latitude=-8.5833&longitude=116.1167&start_date=2024-01-01&end_date=2024-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Cwind_speed_10m_max&timezone=Asia%2FMakassar
File JSON mentah: /content/UTS-DAM-Pariwisata-NTB/data/raw/open_meteo_mataram_2024_raw.json


In [25]:
weather = pd.DataFrame(api_json['daily']).rename(columns={
    'time':'tanggal',
    'temperature_2m_mean':'suhu_rata_rata_c',
    'precipitation_sum':'curah_hujan_mm',
    'wind_speed_10m_max':'kecepatan_angin_maks_kmh'
})
weather['tanggal'] = pd.to_datetime(weather['tanggal'])
weather['tahun'] = weather['tanggal'].dt.year
weather['bulan_angka'] = weather['tanggal'].dt.month
if len(weather) < 30:
    raise ValueError(f'API menghasilkan {len(weather)} baris; perlu minimal 30.')
weather.to_csv(PROCESSED/'data_api_cuaca_harian.csv', index=False)
print('Jumlah baris API:', len(weather))
display(weather.head())


Jumlah baris API: 366


,tanggal,suhu_rata_rata_c,curah_hujan_mm,kecepatan_angin_maks_kmh,tahun,bulan_angka
0,2024-01-01,27.0,6.3,12.2,2024,1
1,2024-01-02,27.6,5.7,11.4,2024,1
2,2024-01-03,27.7,3.8,13.2,2024,1
3,2024-01-04,27.5,4.9,14.1,2024,1
4,2024-01-05,26.4,3.5,8.6,2024,1


In [27]:
# Definisikan URL target sebelum melakukan pengecekan robots.txt
BASE_URL = 'https://data.ntbprov.go.id'
ROBOTS_URL = f'{BASE_URL}/robots.txt'
SCRAPE_URL = f'{BASE_URL}/dataset/jumlah-kunjungan-wisatawan-ke-provinsi-ntb'

robots_status = None
robots_allowed = False
robots_text = ''
scrape_status = None
scrape_rows = 0
scrape_note = ''
try:
    time.sleep(1)
    robots_response = request_with_retry(ROBOTS_URL, headers=headers, timeout=25)
    robots_status = robots_response.status_code
    robots_text = robots_response.text
    (META/'robots_ntb.txt').write_text(robots_text, encoding='utf-8')
    robot_parser = RobotFileParser()
    robot_parser.parse(robots_text.splitlines())
    robots_allowed = robot_parser.can_fetch(USER_AGENT, SCRAPE_URL)
except Exception as exc:
    scrape_note = f'robots.txt tidak dapat diverifikasi ({exc}); scraping dihentikan demi kepatuhan.'
print('Robots.txt HTTP status:', robots_status)
print('Target diizinkan:', robots_allowed)
print('Target:', SCRAPE_URL)
print('Catatan:', scrape_note)

Robots.txt HTTP status: 200
Target diizinkan: True
Target: https://data.ntbprov.go.id/dataset/jumlah-kunjungan-wisatawan-ke-provinsi-ntb
Catatan: 


In [29]:
if robots_allowed:
    time.sleep(2)
    try:
        page_response = request_with_retry(SCRAPE_URL, headers=headers, timeout=30)
        scrape_status = page_response.status_code
        (RAW/'halaman_satu_data_ntb_raw.html').write_text(page_response.text, encoding='utf-8')
        soup = BeautifulSoup(page_response.text, 'html.parser')

        # Ekstrak tabel jika tersedia; jika tidak, ekstrak daftar tautan dataset yang tampak.
        try:
            tables = pd.read_html(page_response.text)
        except ValueError:
            tables = []
        if tables:
            scraped = pd.concat(tables, ignore_index=True)
            scrape_note = f'Mengambil {len(tables)} tabel HTML.'
        else:
            records = []
            for a in soup.select('a[href]'):
                label = a.get_text(' ', strip=True)
                href = a.get('href')
                if label and href:
                    records.append({'teks_tautan':label, 'url':requests.compat.urljoin(SCRAPE_URL, href)})
            scraped = pd.DataFrame(records).drop_duplicates()
            scrape_note = 'Tidak ada tabel HTML; mengekstrak listing tautan halaman.'
        if scraped.empty:
            scrape_note += ' Tidak ditemukan baris yang dapat diekstrak.'
        else:
            scraped.to_csv(RAW/'hasil_scraping_satu_data_ntb.csv', index=False)
            scrape_rows = len(scraped)
        display(scraped.head(10))
    except requests.RequestException as exc:
        scrape_status = getattr(exc.response, 'status_code', 500)
        scrape_note = f'Gagal mengunduh halaman (HTTP {scrape_status}): {exc}'
        print(scrape_note)
else:
    scrape_note = 'Scraping tidak dijalankan karena robots.txt tidak mengizinkan URL target. Pilih halaman statis lain yang diizinkan; jangan bypass robots.txt.'
    print(scrape_note)

scrape_evidence = pd.DataFrame([{
    'robots_url':ROBOTS_URL, 'robots_http_status':robots_status,
    'target_url':SCRAPE_URL, 'robots_allowed':robots_allowed,
    'scrape_http_status':scrape_status, 'jumlah_baris_hasil':scrape_rows,
    'catatan':scrape_note
}])
scrape_evidence.to_csv(META/'bukti_robots_scraping.csv', index=False)
display(scrape_evidence)

Gagal mengunduh halaman (HTTP 404): 404 Client Error: Not Found for url: https://data.ntbprov.go.id/dataset/jumlah-kunjungan-wisatawan-ke-provinsi-ntb


,robots_url,robots_http_status,target_url,robots_allowed,scrape_http_status,jumlah_baris_hasil,catatan
0,https://data.ntbprov.go.id/robots.txt,200,https://data.ntbprov.go.id/dataset/jumlah-kunj...,True,404,0,Gagal mengunduh halaman (HTTP 404): 404 Client...


In [30]:
month_map = {'januari':1,'februari':2,'maret':3,'april':4,'mei':5,'juni':6,
             'juli':7,'agustus':8,'september':9,'oktober':10,'november':11,'desember':12}
analysis = df.copy()
analysis['bulan_angka'] = analysis['bulan'].astype(str).str.lower().map(month_map)
analysis['tahun'] = pd.to_numeric(analysis['tahun'], errors='coerce').astype('Int64')
visitor_2024 = analysis.loc[analysis['tahun'].eq(2024)].dropna(subset=['bulan_angka','jumlah']).copy()
visitor_2024['bulan_angka'] = visitor_2024['bulan_angka'].astype(int)

weather_monthly = weather.groupby(['tahun','bulan_angka'], as_index=False).agg(
    suhu_rata_rata_c=('suhu_rata_rata_c','mean'),
    curah_hujan_bulanan_mm=('curah_hujan_mm','sum'),
    kecepatan_angin_maks_kmh=('kecepatan_angin_maks_kmh','max')
)
print('Baris wisatawan tahun 2024:', len(visitor_2024))
print('Baris cuaca bulanan:', len(weather_monthly))
print('Kunci duplikat pada cuaca:', weather_monthly.duplicated(['tahun','bulan_angka']).sum())

if visitor_2024.empty:
    print('PERHATIAN: tidak ada record wisatawan tahun 2024 pada dataset. Tidak ada merge yang dapat dibuat.')
    merged = pd.DataFrame()
else:
    merged = visitor_2024.merge(
        weather_monthly,
        on=['tahun','bulan_angka'],
        how='left',
        validate='many_to_one',
        indicator=True
    )
    print('Status hasil merge:')
    display(merged['_merge'].value_counts(dropna=False))
    print('Jumlah baris sebelum merge:', len(visitor_2024))
    print('Jumlah baris setelah merge:', len(merged))
    print('Duplikasi baris penuh setelah merge:', int(merged.duplicated().sum()))
    assert len(merged) == len(visitor_2024), 'Jumlah baris berubah; periksa kunci merge.'
    merged.to_csv(PROCESSED/'hasil_gabungan_wisatawan_dan_cuaca_2024.csv', index=False)
    display(merged.head(10))

Baris wisatawan tahun 2024: 24
Baris cuaca bulanan: 12
Kunci duplikat pada cuaca: 0
Status hasil merge:


,count
_merge,
both,24
left_only,0
right_only,0


Jumlah baris sebelum merge: 24
Jumlah baris setelah merge: 24
Duplikasi baris penuh setelah merge: 0


,nomor,kode_provinsi,nama_provinsi,kategori_wisatawan,jumlah_orang,tahun,bulan,jumlah,bulan_angka,suhu_rata_rata_c,curah_hujan_bulanan_mm,kecepatan_angin_maks_kmh,_merge
0,37,52,Nusa Tenggara Barat,Wisatawan Nusantara,"109,025",2024,Desember,109025,12,26.122581,402.5,20.6,both
1,38,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"66,821",2024,Desember,66821,12,26.122581,402.5,20.6,both
2,39,52,Nusa Tenggara Barat,Wisatawan Nusantara,"98,972",2024,November,98972,11,27.096667,254.0,22.1,both
3,40,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"60,660",2024,November,60660,11,27.096667,254.0,22.1,both
4,41,52,Nusa Tenggara Barat,Wisatawan Nusantara,"120,866",2024,Oktober,120866,10,28.141935,51.7,22.7,both
5,42,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"74,079",2024,Oktober,74079,10,28.141935,51.7,22.7,both
6,43,52,Nusa Tenggara Barat,Wisatawan Nusantara,"142,728",2024,September,142728,9,27.203333,59.0,21.9,both
7,44,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"87,478",2024,September,87478,9,27.203333,59.0,21.9,both
8,45,52,Nusa Tenggara Barat,Wisatawan Nusantara,"157,633",2024,Agustus,157633,8,26.129032,11.7,25.0,both
9,46,52,Nusa Tenggara Barat,Wisatawan Mancanegara,"96,613",2024,Agustus,96613,8,26.129032,11.7,25.0,both


In [31]:
def md5_file(path):
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

accessed = datetime.now().astimezone().isoformat(timespec='seconds')
provenance_rows = []

provenance_rows.append({
    'sumber':'Dataset kunjungan wisatawan NTB',
    'url':'https://data.ntbprov.go.id/dataset/9e270f88-a604-4640-a4f0-24c996122b92/show',
    'tanggal_akses':accessed,
    'lisensi':'License Not Specified pada metadata portal; perlu konfirmasi',
    'file':'data/raw/dataset_kunjungan_wisatawan_ntb_asli.xlsx',
    'md5':md5_file(dataset_path)
})
provenance_rows.append({
    'sumber':'Open-Meteo Historical Weather API',
    'url':api_response.url,
    'tanggal_akses':accessed,
    'lisensi':'CC BY 4.0; atribusi Open-Meteo wajib (https://open-meteo.com/)' ,
    'file':'data/raw/open_meteo_mataram_2024_raw.json',
    'md5':md5_file(api_raw_path)
})
html_path = RAW/'halaman_satu_data_ntb_raw.html'
if html_path.exists():
    provenance_rows.append({
        'sumber':'Halaman detail Satu Data NTB (scraping)',
        'url':SCRAPE_URL,
        'tanggal_akses':accessed,
        'lisensi':'License Not Specified pada portal; atribusikan sumber',
        'file':'data/raw/halaman_satu_data_ntb_raw.html',
        'md5':md5_file(html_path)
    })

provenance = pd.DataFrame(provenance_rows)
display(provenance)
provenance.to_csv(META/'provenance.csv', index=False)

,sumber,url,tanggal_akses,lisensi,file,md5
0,Dataset kunjungan wisatawan NTB,https://data.ntbprov.go.id/dataset/9e270f88-a6...,2026-11-26T18:54:59+00:00,License Not Specified pada metadata portal; pe...,data/raw/dataset_kunjungan_wisatawan_ntb_asli....,f17c3f11b70aad5bac001c90e0353305
1,Open-Meteo Historical Weather API,https://archive-api.open-meteo.com/v1/archive?...,2026-11-26T18:54:59+00:00,CC BY 4.0; atribusi Open-Meteo wajib (https://...,data/raw/open_meteo_mataram_2024_raw.json,d6591a1a015726b66e7d7dcf5eed7395
